In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# GPU kontrolü
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Kullanılan cihaz: {device}")  # cuda çıkmalı

c:\Users\ENES\Desktop\SpamKiller\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Kullanılan cihaz: cuda


In [2]:
df = pd.read_csv('../data/spam_islenmis.csv')
df = df.dropna(subset=['metin_tr'])

X = df['metin_tr'].tolist()   # Ham Türkçe metin (temizlenmiş değil, BERTurk kendi anlıyor)
y = df['label'].tolist()

# Eğitim/test böl
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Eğitim: {len(X_train)} | Test: {len(X_test)}")

Eğitim: 4456 | Test: 1114


In [3]:
MODEL_ADI = "dbmdz/bert-base-turkish-cased"

print("Tokenizer indiriliyor...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ADI)
print("Tokenizer hazır ✓")

# Test
ornek = tokenizer("Merhaba, bu bir test cümlesidir.", 
                   return_tensors='pt', truncation=True, max_length=128)
print(ornek)

Tokenizer indiriliyor...


c:\Users\ENES\Desktop\SpamKiller\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ENES\.cache\huggingface\hub\models--dbmdz--bert-base-turkish-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokenizer hazır ✓
{'input_ids': tensor([[    2,  6965,    16,  2048,  1996,  4914, 30773,  2067,    18,     3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [4]:
class SpamDataset(Dataset):
    def __init__(self, metinler, etiketler, tokenizer, max_len=128):
        self.metinler = metinler
        self.etiketler = etiketler
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.metinler)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.metinler[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.etiketler[idx], dtype=torch.long)
        }

# Dataset ve DataLoader oluştur
train_dataset = SpamDataset(X_train, y_train, tokenizer)
test_dataset  = SpamDataset(X_test,  y_test,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"Eğitim batch sayısı: {len(train_loader)}")
print(f"Test batch sayısı:   {len(test_loader)}")

Eğitim batch sayısı: 140
Test batch sayısı:   35


In [5]:
print("BERTurk modeli indiriliyor...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ADI, num_labels=2)
model = model.to(device)
print("Model GPU'ya yüklendi ✓")

optimizer = AdamW(model.parameters(), lr=2e-5)

BERTurk modeli indiriliyor...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7888.95it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

Model GPU'ya yüklendi ✓


In [6]:
EPOCH_SAYISI = 3  # 3 tur yeterli

for epoch in range(EPOCH_SAYISI):
    model.train()
    toplam_kayip = 0

    for i, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        kayip = outputs.loss
        kayip.backward()
        optimizer.step()

        toplam_kayip += kayip.item()

        # Her 20 batch'te bir ilerleme göster
        if (i + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{EPOCH_SAYISI} | Batch {i+1}/{len(train_loader)} | Kayıp: {kayip.item():.4f}")

    ort_kayip = toplam_kayip / len(train_loader)
    print(f"\n✓ Epoch {epoch+1} tamamlandı | Ortalama Kayıp: {ort_kayip:.4f}\n")

print("Eğitim tamamlandı! 🎉")

Epoch 1/3 | Batch 20/140 | Kayıp: 0.3112
Epoch 1/3 | Batch 40/140 | Kayıp: 0.1624
Epoch 1/3 | Batch 60/140 | Kayıp: 0.0202
Epoch 1/3 | Batch 80/140 | Kayıp: 0.0779
Epoch 1/3 | Batch 100/140 | Kayıp: 0.0093
Epoch 1/3 | Batch 120/140 | Kayıp: 0.0188
Epoch 1/3 | Batch 140/140 | Kayıp: 0.0064

✓ Epoch 1 tamamlandı | Ortalama Kayıp: 0.1207

Epoch 2/3 | Batch 20/140 | Kayıp: 0.0041
Epoch 2/3 | Batch 40/140 | Kayıp: 0.0378
Epoch 2/3 | Batch 60/140 | Kayıp: 0.0058
Epoch 2/3 | Batch 80/140 | Kayıp: 0.1261
Epoch 2/3 | Batch 100/140 | Kayıp: 0.0124
Epoch 2/3 | Batch 120/140 | Kayıp: 0.0048
Epoch 2/3 | Batch 140/140 | Kayıp: 0.0102

✓ Epoch 2 tamamlandı | Ortalama Kayıp: 0.0291

Epoch 3/3 | Batch 20/140 | Kayıp: 0.0027
Epoch 3/3 | Batch 40/140 | Kayıp: 0.0110
Epoch 3/3 | Batch 60/140 | Kayıp: 0.0010
Epoch 3/3 | Batch 80/140 | Kayıp: 0.0009
Epoch 3/3 | Batch 100/140 | Kayıp: 0.0049
Epoch 3/3 | Batch 120/140 | Kayıp: 0.0017
Epoch 3/3 | Batch 140/140 | Kayıp: 0.0012

✓ Epoch 3 tamamlandı | Ortalama K

In [7]:
model.eval()
tum_tahminler = []
tum_gercekler = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        tahminler = torch.argmax(outputs.logits, dim=1)

        tum_tahminler.extend(tahminler.cpu().numpy())
        tum_gercekler.extend(labels.cpu().numpy())

print("="*40)
print("  BERTurk Sonuçları")
print("="*40)
print(f"Accuracy  : %{accuracy_score(tum_gercekler, tum_tahminler)*100:.2f}")
print(f"Precision : %{precision_score(tum_gercekler, tum_tahminler)*100:.2f}")
print(f"Recall    : %{recall_score(tum_gercekler, tum_tahminler)*100:.2f}")
print(f"F1 Score  : %{f1_score(tum_gercekler, tum_tahminler)*100:.2f}")

  BERTurk Sonuçları
Accuracy  : %98.56
Precision : %93.46
Recall    : %95.97
F1 Score  : %94.70


In [8]:
model.save_pretrained('../models/berturk_model')
tokenizer.save_pretrained('../models/berturk_model')
print("BERTurk kaydedildi ✓")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.61it/s]

BERTurk kaydedildi ✓
